# 04_pipeline_runner
Runs the pipeline for one date: 02_raw_to_bronze, then 03_bronze_to_silver. Stops and logs a Failure row if a step breaks.
Parameters: load_type (full / incremental), date (YYYY-MM-DD).

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("date", "")

In [0]:
import uuid
from datetime import datetime, date as _date
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

CATALOG, SCHEMA = "workspace", "steam"
LAYER = "Pipeline-Runner"
TIMEOUT_SECONDS = 3600
spark.sql(f"USE {CATALOG}.{SCHEMA}")

load_type = dbutils.widgets.get("load_type")
run_date = dbutils.widgets.get("date").strip() or str(_date.today())
datetime.strptime(run_date, "%Y-%m-%d")  # fails fast on a bad date
run_id = str(uuid.uuid4())

# each step: (notebook, parameters passed to it)
STEPS = [
    ("02_raw_to_bronze", {"load_type": load_type, "date": run_date, "landing_path": ""}),
    ("03_bronze_to_silver", {"date": run_date, "batch_id": ""}),
]
print(load_type, run_date, run_id)

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("file_processed", StringType()),
    StructField("batch_id", StringType()), StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()), StructField("status", StringType()),
    StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_skipped_unchanged", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

def write_log(start, status, error=None):
    """One row for the whole pipeline run. Row counts live on each step's own log row."""
    row = (run_id, LAYER, load_type, f"date={run_date}", None, start, datetime.now(),
           status, None, None, None, None, error)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("pipeline_execution_logs")

In [0]:
start = datetime.now()
current = None
try:
    for notebook, params in STEPS:
        current = notebook
        print(f"running {notebook} ...")
        dbutils.notebook.run(f"./{notebook}", TIMEOUT_SECONDS, params)
        print(f"{notebook} finished")
    write_log(start, "Success")
    print("pipeline finished")
except Exception as e:
    write_log(start, "Failure", f"{current} failed: {str(e)[:1900]}")
    raise

In [0]:
display(spark.sql("""
    SELECT layer, load_type, file_processed, status, rows_inserted, rows_updated,
           rows_skipped_unchanged, rows_quarantined, error_message, start_time
    FROM pipeline_execution_logs ORDER BY start_time DESC LIMIT 6
"""))
display(spark.sql("""
    SELECT (SELECT COUNT(*) FROM bronze_steam) AS bronze_rows,
           (SELECT COUNT(*) FROM silver_steam) AS silver_rows
"""))